In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# 1. 加载数据
print("正在加载数据...")
# 修正文件路径
df_original = pd.read_excel('../../data\quyi_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
# Paper protocol: preserve furnace order; first 900 records train, next 500 test.
if len(X_original) != 1400 or len(y_original) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
X_train, X_test = X_original.iloc[:900].copy(), X_original.iloc[900:].copy()
y_train, y_test = y_original.iloc[:900].copy(), y_original.iloc[900:].copy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化

# 最优参数
# 最优参数:
# n_estimators: 500
# max_depth: 25
# min_samples_split: 2
# min_samples_leaf: 1
# max_features: None

# 6. 创建模型
base_rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=25,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    n_jobs=-1,
    random_state=42
)
model = MultiOutputRegressor(base_rf)

# 7. 执行K折交叉验证
print("\n开始执行交叉验证...")
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

cv_scores = {var: [] for var in target_vars}
cv_rmse = {var: [] for var in target_vars}
cv_mae = {var: [] for var in target_vars}

for train_idx, val_idx in kf.split(X_train):
    # 分割训练集和验证集
    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # 训练模型
    model.fit(X_train_fold, y_train_fold)
    
    # 在验证集上预测
    y_val_pred = model.predict(X_val_fold)
    
    # 计算每个目标变量的指标
    for i, var in enumerate(target_vars):
        r2 = r2_score(y_val_fold.iloc[:, i], y_val_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_val_fold.iloc[:, i], y_val_pred[:, i]))
        mae = mean_absolute_error(y_val_fold.iloc[:, i], y_val_pred[:, i])
        
        cv_scores[var].append(r2)
        cv_rmse[var].append(rmse)
        cv_mae[var].append(mae)

# 打印交叉验证结果
print(f"\n{k_folds}折交叉验证结果:")
mean_cv_r2 = 0
mean_cv_rmse = 0
mean_cv_mae = 0

for var in target_vars:
    mean_r2 = np.mean(cv_scores[var])
    mean_rmse = np.mean(cv_rmse[var])
    mean_mae = np.mean(cv_mae[var])
    
    mean_cv_r2 += mean_r2
    mean_cv_rmse += mean_rmse
    mean_cv_mae += mean_mae
    
    print(f"\n{var} 交叉验证指标:")
    print(f"平均 R²: {mean_r2:.4f} ± {np.std(cv_scores[var]):.4f}")
    print(f"平均 RMSE: {mean_rmse:.4f} ± {np.std(cv_rmse[var]):.4f}")
    print(f"平均 MAE: {mean_mae:.4f} ± {np.std(cv_mae[var]):.4f}")

# 计算所有目标变量的平均指标
mean_cv_r2 /= len(target_vars)
mean_cv_rmse /= len(target_vars)
mean_cv_mae /= len(target_vars)

print("\n所有变量平均交叉验证指标:")
print(f"平均 R²: {mean_cv_r2:.4f}")
print(f"平均 RMSE: {mean_cv_rmse:.4f}")
print(f"平均 MAE: {mean_cv_mae:.4f}")

# 8. 在完整训练集上训练最终模型
print("\n开始训练最终随机森林模型...")
model.fit(X_train, y_train)
print("模型训练完成！")

# 9. 在测试集上进行预测
y_pred = model.predict(X_test)

# 10. 评估最终模型性能
print("\n最终模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")
    print('-----------------------------------')

# 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")